# Data measurement of the two-point energy correlator observable

This notebook implements the data measurement of the two-point energy correlator observable (EEC) using the ATLAS Omnifold Z+jets measurement. The result is compared to MadGraph+Pythia8 and Sherpa theory predictions.

See `eec_pseudo_results.ipynb` for the pseudodata-based closure test that validates the measurement method.

Note that running this notebook over all events in the samples is expensive!
Unless you have access to significant resources it is recommended to set `max_events` to an integer (default set to `None` in which case all events are used).
The histogramming of pairs of tracks is the computational bottleneck in this measurement.
It is accelerated using the `boost_histogram` python package and parallelized with dask.
See `eec_utils.py` for details.

## Preliminaries: load all data

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys, os
# Disable HDF5 file locking to avoid issues on the NERSC filesystems
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"
import pickle
import numpy as np
import pandas as pd
import dask.distributed as distributed

import eec_utils as eec
sys.path.insert(0, os.path.abspath(".."))
import utils.visualize as vis

In [ ]:
# ROOT file paths
staging_dir = "/global/cfs/cdirs/m3246/ZjetOmnifold/data/final_public_data"
tree_name = "OmniTree"

truth_mc_path = f"{staging_dir}/truth_mc_events.root"
truth_pseudodata_path = f"{staging_dir}/truth_pseudodata_events.root"
truth_hv_path = f"{staging_dir}/truth_mc_hv_events.root"
truth_madgraph_path = f"{staging_dir}/truth_madgraph/truth_madgraph_events.root"
truth_sherpa_path = f"{staging_dir}/truth_sherpa/truth_sherpa_events.root"

In [ ]:
# Load event weights for data measurement
data_weights = pd.read_hdf(f"{staging_dir}/data_weights.h5")
data_hv_weights = pd.read_hdf(f"{staging_dir}/data_hv_weights.h5")
truth_madgraph_weights = pd.read_hdf(f"{staging_dir}/truth_madgraph/truth_madgraph_weights.h5")
truth_sherpa_weights = pd.read_hdf(f"{staging_dir}/truth_sherpa/truth_sherpa_weights.h5")

## Measurement with data

In [ ]:
# Create dictionaries of event weight vectors needed for the data measurement
nominal_weights = data_weights["weights_nominal"]
n_init_weights = [name for name in data_weights.keys() if "weights_ensemble_" in name]
dbootstrap_weights = [name for name in data_weights.keys() if "weights_bootstrap_data_" in name]
mcbootstrap_train_weights = [
    name for name in data_weights.keys() if "weights_bootstrap_mc_" in name
]
data_weights_dict = {
    "nominal": data_weights["weights_nominal"].to_numpy(),
    **{
        f"ensemble_{i}": data_weights[name].to_numpy()
        for i, name in enumerate(n_init_weights)
    },
    **{
        f"bootstrap_data_{i}": data_weights[name].to_numpy()
        for i, name in enumerate(dbootstrap_weights)
    },
    **{
        f"bootstrap_mc_{i}": data_weights[name].to_numpy()
        for i, name in enumerate(mcbootstrap_train_weights)
    },
    "trackEffMain": data_weights["weights_trackEffMain"].to_numpy(),
    "trackEffJet": data_weights["weights_trackEffJet"].to_numpy(),
    "trackFake": data_weights["weights_trackFake"].to_numpy(),
    "trackPtScale": data_weights["weights_trackPtScale"].to_numpy(),
    "muCalID": data_weights["weights_muCalID"].to_numpy(),
    "muCalMS": data_weights["weights_muCalMS"].to_numpy(),
    "muCalResBias": data_weights["weights_muCalResBias"].to_numpy(),
    "muCalScale": data_weights["weights_muCalScale"].to_numpy(),
    "muEffReco": data_weights["weights_muEffReco"].to_numpy(),
    "muEffIso": data_weights["weights_muEffIso"].to_numpy(),
    "muEffTrack": data_weights["weights_muEffTrack"].to_numpy(),
    "muEffTrig": data_weights["weights_muEffTrig"].to_numpy(),
    "pileup": data_weights["weights_pileup"].to_numpy(),
    "lumi": data_weights["weights_lumi"].to_numpy(),
    "theoryQCD": data_weights["weights_theoryQCD"].to_numpy(),
    "theoryPDF": data_weights["weights_theoryPDF"].to_numpy(),
    "theoryAlphaS": data_weights["weights_theoryAlphaS"].to_numpy(),
    "theoryPSsoft": data_weights["weights_theoryPSsoft"].to_numpy(),
    "theoryPSjet": data_weights["weights_theoryPSjet"].to_numpy(),
    "theoryPSscale": data_weights["weights_theoryPSscale"].to_numpy(),
    "theoryMPI": data_weights["weights_theoryMPI"].to_numpy(),
    "topBackground": data_weights["weights_topBackground"].to_numpy(),
    "nonstrongDiboson": data_weights["weights_nonstrongDiboson"].to_numpy(),
    "nonstrongEW": data_weights["weights_nonstrongEW"].to_numpy(),
    "dd": data_weights["weights_dd"].to_numpy(),
    "target_dd": data_weights["target_dd"].to_numpy(),
    "hvhad": data_weights["weights_hvhad"].to_numpy(),
}
data_hv_weights_dict = {
    "hv": data_hv_weights["weights_hv"].to_numpy(),
}
truth_madgraph_weights_dict = {
    "nominal": truth_madgraph_weights["weights_nominal"].to_numpy(),
    "theoryQCD": truth_madgraph_weights["weights_theoryQCD"].to_numpy(),
    "theoryPDF": truth_madgraph_weights["weights_theoryPDF"].to_numpy(),
    "theoryAlphaS": truth_madgraph_weights["weights_theoryAlphaS"].to_numpy(),
    "theoryPSsoft": truth_madgraph_weights["weights_theoryPSsoft"].to_numpy(),
    "theoryPSjet": truth_madgraph_weights["weights_theoryPSjet"].to_numpy(),
    "theoryPSscale": truth_madgraph_weights["weights_theoryPSscale"].to_numpy(),
    "theoryMPI": truth_madgraph_weights["weights_theoryMPI"].to_numpy(),
}
truth_sherpa_weights_dict = {
    "nominal": truth_sherpa_weights["weights_nominal"].to_numpy(),
    "theoryQCD": truth_sherpa_weights["weights_theoryQCD"].to_numpy(),
    "theoryPDF": truth_sherpa_weights["weights_theoryPDF"].to_numpy(),
    "theoryAlphaS": truth_sherpa_weights["weights_theoryAlphaS"].to_numpy(),
}

In [ ]:
bins = np.logspace(-6, 0, 40)

In [ ]:
# Run computations with dask
n_cores = os.cpu_count()
cluster = distributed.LocalCluster(
    n_workers=128,
    threads_per_worker=2,
    memory_limit="auto",
)
client = distributed.Client(cluster)

try:
    print("Histogramming Madgraph MC sample")
    truth_mc_hists = eec.run_eec_workflow_parallel(
        truth_mc_path,
        "OmniTree",
        data_weights_dict,
        bins,
        chunk_size=1000,
        invert_z=False,
    )
    print("Histogramming HV sample")
    truth_hv_hists = eec.run_eec_workflow_parallel(
        truth_hv_path,
        "OmniTree",
        data_hv_weights_dict,
        bins,
        chunk_size=1000,
        invert_z=False,
    )
    print("Histogramming Madgraph prediction")
    truth_madgraph_hists = eec.run_eec_workflow_parallel(
        truth_madgraph_path,
        "OmniTree",
        truth_madgraph_weights_dict,
        bins,
        chunk_size=1000,
        invert_z=False,
    )
    print("Histogramming Sherpa prediction")
    truth_sherpa_hists = eec.run_eec_workflow_parallel(
        truth_sherpa_path,
        "OmniTree",
        truth_sherpa_weights_dict,
        bins,
        chunk_size=1000,
        invert_z=False,
    )
finally:
    client.close()
    cluster.close()

In [ ]:
# Pickle the resulting histograms
hist_dir = "./hist_storage"
with open(os.path.join(hist_dir, "data_eec_hists.pkl"), "wb") as f:
    pickle.dump(
        {
            "truth_mc": truth_mc_hists,
            "truth_hv": truth_hv_hists,
            "truth_madgraph": truth_madgraph_hists,
            "truth_sherpa": truth_sherpa_hists,
        },
        f,
    )

## Plotting

In [ ]:
# Load the histograms
hist_dir = "./hist_storage"
with open(os.path.join(hist_dir, "data_eec_hists.pkl"), "rb") as f:
    all_hists = pickle.load(f)

measurement_hists = {**all_hists["truth_mc"], **all_hists["truth_hv"]}
truth_madgraph_hists = all_hists["truth_madgraph"]
truth_sherpa_hists = all_hists["truth_sherpa"]

In [ ]:
# Plot measurement with full uncertainties
meas, budget, cov = vis.draw_plot(
    measurement_hists,
    bins,
    ylabel="EEC",
    xlabel="z",
    logyScale=True,
    logxScale=True,
    is_xSec=False,
    is_omni_data=True,
    ratio_ylim=[0.7, 1.3],
    mgfxfx_truth_results=truth_madgraph_hists,
    sherpa_truth_results=truth_sherpa_hists,
    draw_uncertainty_budget=True,
    draw_cov_matrix=True,
    pdf_name="plot_storage/data_eec_xsec.pdf",
)
meas.show()
budget.show()
cov.show()